WhatNet  –  Arabic

In [ ]:
#  Arabic is a cursive abjad script written right-to-left.  Characters change
#  shape depending on their position in a word (isolated, initial, medial,
#  final), and many letters are distinguished solely by dot diacritics above or
#  below the base form.
#
#    • num_classes   : 28   (AHCD: 28 Arabic letters × isolated forms
#                                  + sub-letter variants as catalogued in the
#                                  dataset; exact count may vary by release –
#                                  a runtime warning fires on mismatch)
#    • image_size    : 32   (AHCD images are 32×32 greyscale)
#    • Dataset format: .npz arrays
#                      → folder-per-class PNG tree  (same layout as DHCD)
#    • Dataset loader: npz → Pillow file-walker (identical to Bengali version)
#    • Scaffold branch: 1×3 (Kannada short horizontals) → 3×3 + diagonal pair
#                       (3×3 captures connected cursive base strokes;
#                        a 3×3 dilated branch captures the dot diacritics
#                        which sit above/below the baseline at ~3-px offsets)
#     Decoder Head (Multi-scale GAP, AFC, Classification Head, Gated Fusion)
#    • Augmentation  : random_flip_left_right added – Arabic writers vary in
#                      pen pressure and slight lateral tilt, and the isolated-
#                      form dataset is already normalised for orientation, so
#                      mild horizontal flip generalises across writers without
#                      creating invalid glyphs (unlike vertical flip).

#  Dataset
#  AHCD – Arabic Handwritten Characters Dataset  (El-Sherif & Abdelazim, 2017)
#    • 16 800 images, 32×32 greyscale, folder-per-class layout
#    • 66 classes covering isolated forms of the 28 Arabic letters
#    • Kaggle:
#        https://www.kaggle.com/datasets/mloey1/ahcd1
#    • Expected directory structure after extraction:
#        data_dir/
#          Train/
#            1/  *.png   (class label 1 = ا  alef)
#            2/  *.png
#            …
#            66/ *.png
#          Test/
#            1/  *.png
#            …
#            66/ *.png

In [ ]:
# importing necessary dependencies
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from PIL import Image as PilImage
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

#  0. REPRODUCIBILITY
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

#  2. DATASET PIPELINE  — CSV loader replaces the Pillow file-walker
import glob, os
import numpy as np
import tensorflow as tf

#  0. REPRODUCIBILITY
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

#  1. CONFIGURATION
CFG = {
    #   28 classes for AHCD (28 Arabic letters, isolated forms).
    #   Some Kaggle releases label 1-indexed folders 1–66.  The loader sorts
    #   folder names lexicographically and assigns 0-based indices, so the
    #   mapping is consistent regardless of the naming convention used.
    #   If your copy of the dataset has a different number of class folders,
    #   a runtime [WARN] is printed and you should update this value.
    "num_classes":      28,
    "image_size":       32,
    "batch_size":       64,
    "epochs":           100,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.1,
    "val_split":        0.1,

    #   root of the AHCD folder-per-class tree.
    #   Kaggle notebook path (after adding the dataset):
    #     "/kaggle/input/ahcd1"
    #   Expected layout:  data_dir/Train/<class_folder>/*.png
    #                     data_dir/Test/<class_folder>/*.png
    "data_dir":    "/kaggle/input/datasets/mloey1/ahcd1/",
    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE


def _find_csv(data_dir: str, pattern: str) -> str:
    """
    Locate a CSV file whose name contains `pattern` (case-insensitive).
    Raises a clear FileNotFoundError if nothing matches.
    """
    candidates = [
        p for p in glob.glob(os.path.join(data_dir, "*.csv"))
        if pattern.lower() in os.path.basename(p).lower()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"[ERROR] No CSV matching '*{pattern}*' found in {data_dir}\n"
            f"  Files present: {os.listdir(data_dir)}"
        )
    # Prefer the shortest name (most direct match)
    return sorted(candidates, key=len)[0]


def _load_csv_dataset(data_dir: str, split: str):
    """
    Load one split ('Train' or 'Test') from the flat CSV files.

    Returns:
    images : float32 ndarray, shape (N, 32, 32, 1), values in [0, 255]
    labels : int32 ndarray,   shape (N,),            values in [0, 27]
    """
    img_path = _find_csv(data_dir, f"csv{split}Images")
    lbl_path = _find_csv(data_dir, f"csv{split}Label")

    print(f"[INFO] Loading {split} images from : {os.path.basename(img_path)}")
    print(f"[INFO] Loading {split} labels from : {os.path.basename(lbl_path)}")

    # Read raw arrays — no header row in either file
    images_flat = np.loadtxt(img_path, delimiter=",", dtype=np.float32)  # (N, 1024)
    labels_raw  = np.loadtxt(lbl_path, delimiter=",", dtype=np.int32)    # (N,) or (N, 1)

    labels_raw = labels_raw.ravel()           # ensure 1-D
    labels     = labels_raw - 1              # shift 1-based -> 0-based  [0, 27]

    # Reshape: (N, 1024) -> (N, 32, 32, 1)
    N = images_flat.shape[0]
    images = images_flat.reshape(N, IMG, IMG, 1)

    # Sanity checks
    n_classes = int(labels.max()) + 1
    print(f"[INFO] {split}: {N:,} samples, {n_classes} classes detected "
          f"(label range {labels.min()}–{labels.max()})")
    if n_classes != CFG["num_classes"]:
        print(f"[WARN] Label range implies {n_classes} classes but "
              f"CFG['num_classes']={CFG['num_classes']} — update CFG to match.")

    return images, labels

# Load raw arrays
print("[INFO] Building CSV-backed dataset (AHCD – Arabic) …")
train_images, train_labels = _load_csv_dataset(CFG["data_dir"], "Train")
test_images,  test_labels  = _load_csv_dataset(CFG["data_dir"], "Test")

# Train / val split
N_train_full = train_images.shape[0]
n_val        = max(1, int(N_train_full * CFG["val_split"]))
n_train      = N_train_full - n_val

# Shuffle before splitting so val isn't just the last N samples
rng   = np.random.default_rng(SEED)
perm  = rng.permutation(N_train_full)
train_images = train_images[perm]
train_labels = train_labels[perm]

tr_imgs, tr_lbls = train_images[:n_train], train_labels[:n_train]
vl_imgs, vl_lbls = train_images[n_train:], train_labels[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {test_images.shape[0]:,}")

NUM_CLASSES = CFG["num_classes"]


# Preprocessing helpers
def normalise(img, lbl):
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    img = tf.image.random_flip_left_right(img)
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)


# Build tf.data pipelines
def _make_ds(images, labels, training: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(images), tf.constant(labels, dtype=tf.int32))
    )
    ds = ds.map(normalise, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(8192, seed=SEED)
    ds = ds.map(to_onehot, num_parallel_calls=AUTOTUNE)
    return ds.batch(BS).prefetch(AUTOTUNE)

train_ds = _make_ds(tr_imgs, tr_lbls, training=True)
val_ds   = _make_ds(vl_imgs, vl_lbls, training=False)

# Test pipeline without one-hot (used by compute_macro_f1)
test_ds = (
    tf.data.Dataset.from_tensor_slices(
        (tf.constant(test_images), tf.constant(test_labels, dtype=tf.int32))
    )
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

# Test pipeline with one-hot (used by model.evaluate)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices(
        (tf.constant(test_images), tf.constant(test_labels, dtype=tf.int32))
    )
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

#  3. DISPLAY UTILITIES
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

#  4. BUILDING BLOCKS
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

#  5. MODEL DEFINITION
#   Scaffold uses 3×3 (isotropic) to handle Arabic cursive strokes and dots.
def build_whatnet_arabic(num_classes: int = 66,
    drop_path_rate: float = 0.05,
    dropout_rate: float = 0.3,
    weight_decay: float = 1e-4,
    head_units: int = 256,
    override_tier: int = None,
    image_size: int = 32) -> Model:
    """
    WhatNet-Arabic: WhatNet adapted for AHCD Arabic character recognition.

    Architecture overview
    Stem (dual-path):
      • Standard 3×3 conv path (overall glyph body)
      • Dot-aware scaffold: 3×3 conv – isotropic kernel captures both the
        cursive base stroke AND nearby dot diacritics in a single receptive
        field; combined with dilated branches in STM for far-dot detection
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (16×16 at 32-px input)
      enc2: 64→128   ( 8× 8)
      enc3: 128→256  ( 4× 4)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Multi-scale GAP
      • Adaptive filter capsule (AFC)
      • Classification Head
      • Gated fusion → final MLP + layer norm → logits
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # Stem
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)

    # scaffold: 1×3 (Kannada) → 3×3 (Arabic isotropic dot/curve)
    s        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # Encoder
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # Decoder Head
    # Multi-scale GAP fusion
    gap1 = layers.GlobalAveragePooling2D(name="gap1")(enc1)
    gap2 = layers.GlobalAveragePooling2D(name="gap2")(enc2)
    gap3 = layers.GlobalAveragePooling2D(name="gap3")(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])

    # Adaptive Filter Capsule (AFC)
    # Projects the fused multi-scale vector into capsule space.
    # Each of the K capsules learns to respond to one character class.
    afc_out = adaptive_filter_capsule(fused_gap, num_classes)   # (B, K)

    # Classification head
    # Dense projection of the raw GAP features (residual path alongside AFC)
    x = layers.Dense(head_units, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation("gelu", name="head_act")(x)
    x = layers.Dense(num_classes, name="head_logits")(x)

    # Gated fusion: AFC scores + dense-head logits
    # A learned scalar gate (per-sample softmax over 2 weights) blends the
    # AFC capsule scores with the plain dense logits.  This lets the model
    # learn how much to trust the capsule routing vs. the direct projection.
    combined = layers.Concatenate(name="gate_input")([x, afc_out])
    gate     = layers.Dense(2, activation="softmax", name="gate")(combined)  # (B, 2)

    # gate[:,0] weights the dense head; gate[:,1] weights the AFC output
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x,gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc"  )([afc_out,gate])

    outputs = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=outputs, name="WhatNet-Arabic")       # ← CHANGED

MODELS_TF = {
    "WhatNet-Arabic": lambda: build_whatnet_arabic(NUM_CLASSES, IMG),  # ← CHANGED
}

#  6. LR SCHEDULE  (unchanged)
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

#  7. TRAINING & EVALUATION HELPERS  (unchanged)
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

#  8. TRAIN + EVALUATE
trained_models  = {}
all_histories   = {}
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs"
         f"  [AHCD – Arabic]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=1),
        # EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=1,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

#  9. FINAL TEST-SET EVALUATION
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

#  10. PERSIST RESULTS
results_path = os.path.join(CFG["results_dir"], "arabic_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "arabic_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] AHCD Arabic benchmark complete.\n", "green", "bold"))

In [ ]:
n_actual = len(np.unique(train_labels))
print(n_actual)  # probably 28, not 66

28


In [ ]:
[INFO] Building CSV-backed dataset (AHCD – Arabic) …
[INFO] Loading Train images from : csvTrainImages 13440x1024.csv
[INFO] Loading Train labels from : csvTrainLabel 13440x1.csv
[INFO] Train: 13,440 samples, 28 classes detected (label range 0–27)
[INFO] Loading Test images from : csvTestImages 3360x1024.csv
[INFO] Loading Test labels from : csvTestLabel 3360x1.csv
[INFO] Test: 3,360 samples, 28 classes detected (label range 0–27)
[INFO] Train: 12,096 | Val: 1,344 | Test: 3,360

══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 100 epochs  [AHCD – Arabic]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Arabic  –  Parameter Summary                        ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d_32       ║  Conv2D               ║              288  ║
║  conv2d_33       ║  Conv2D               ║              288  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  dense_4         ║  Dense                ║              520  ║
║  dense_5         ║  Dense                ║              576  ║
║  conv2d_34       ║  Conv2D               ║            4,096  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_35       ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_36       ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_37       ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_38       ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_39       ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_40       ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  … (truncated)   ║                       ║                   ║
╠══════════════════╩═══════════════════════╩══════════════════╣
║  Trainable params                      :          5,423,110  ║
║  Non-trainable params                  :              8,248  ║
║  Total params                          :          5,431,358  ║
╚══════════════════════════════════════════════════════════════╝

  ▶ Training: WhatNet-Arabic
Epoch 1/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 57s 124ms/step - accuracy: 0.3138 - loss: 2.6181 - val_accuracy: 0.0357 - val_loss: 3.4589
Epoch 2/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - accuracy: 0.8440 - loss: 1.1961 - val_accuracy: 0.0871 - val_loss: 3.2790
Epoch 3/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 92ms/step - accuracy: 0.9128 - loss: 0.9897 - val_accuracy: 0.7619 - val_loss: 1.4965
Epoch 4/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 89ms/step - accuracy: 0.9263 - loss: 0.9410 - val_accuracy: 0.6414 - val_loss: 1.6791
Epoch 5/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9373 - loss: 0.8959 - val_accuracy: 0.6979 - val_loss: 1.4702
Epoch 6/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9492 - loss: 0.8509 - val_accuracy: 0.4405 - val_loss: 2.4534
Epoch 7/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9573 - loss: 0.8263 - val_accuracy: 0.3490 - val_loss: 2.5416
Epoch 8/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - accuracy: 0.9612 - loss: 0.8057 - val_accuracy: 0.9509 - val_loss: 0.8330
Epoch 9/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9697 - loss: 0.7825 - val_accuracy: 0.9278 - val_loss: 0.8768
Epoch 10/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9703 - loss: 0.7659 - val_accuracy: 0.9055 - val_loss: 0.9487
Epoch 11/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - accuracy: 0.9716 - loss: 0.7606 - val_accuracy: 0.9665 - val_loss: 0.7687
Epoch 12/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9797 - loss: 0.7348 - val_accuracy: 0.8921 - val_loss: 0.9814
Epoch 13/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9764 - loss: 0.7419 - val_accuracy: 0.9568 - val_loss: 0.8025
Epoch 14/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9825 - loss: 0.7218 - val_accuracy: 0.9568 - val_loss: 0.7814
Epoch 15/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9801 - loss: 0.7264 - val_accuracy: 0.9345 - val_loss: 0.8180
Epoch 16/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9801 - loss: 0.7207 - val_accuracy: 0.9635 - val_loss: 0.7652
Epoch 17/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9861 - loss: 0.7004 - val_accuracy: 0.9241 - val_loss: 0.8703
Epoch 18/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - accuracy: 0.9806 - loss: 0.7100 - val_accuracy: 0.9740 - val_loss: 0.7311
Epoch 19/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9837 - loss: 0.7038 - val_accuracy: 0.9568 - val_loss: 0.7861
Epoch 20/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9860 - loss: 0.7008 - val_accuracy: 0.9717 - val_loss: 0.7388
Epoch 21/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9841 - loss: 0.6973 - val_accuracy: 0.9628 - val_loss: 0.7645
Epoch 22/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9874 - loss: 0.6912 - val_accuracy: 0.9598 - val_loss: 0.7740
Epoch 23/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9861 - loss: 0.6941 - val_accuracy: 0.9650 - val_loss: 0.7549
Epoch 24/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9869 - loss: 0.6883 - val_accuracy: 0.9286 - val_loss: 0.8644
Epoch 25/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9875 - loss: 0.6906 - val_accuracy: 0.9539 - val_loss: 0.7786
Epoch 26/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 89ms/step - accuracy: 0.9902 - loss: 0.6836 - val_accuracy: 0.9702 - val_loss: 0.7340
Epoch 27/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 89ms/step - accuracy: 0.9915 - loss: 0.6758 - val_accuracy: 0.9420 - val_loss: 0.8067
Epoch 28/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9914 - loss: 0.6744 - val_accuracy: 0.9710 - val_loss: 0.7415
Epoch 29/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 93ms/step - accuracy: 0.9913 - loss: 0.6748 - val_accuracy: 0.9747 - val_loss: 0.7374
Epoch 30/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9904 - loss: 0.6764 - val_accuracy: 0.9368 - val_loss: 0.8506
Epoch 31/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9865 - loss: 0.6871 - val_accuracy: 0.9546 - val_loss: 0.7771
Epoch 32/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9939 - loss: 0.6672 - val_accuracy: 0.9680 - val_loss: 0.7390
Epoch 33/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9897 - loss: 0.6748 - val_accuracy: 0.9598 - val_loss: 0.7596
Epoch 34/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9951 - loss: 0.6620 - val_accuracy: 0.9554 - val_loss: 0.7832
Epoch 35/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9939 - loss: 0.6675 - val_accuracy: 0.9249 - val_loss: 0.8397
Epoch 36/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9923 - loss: 0.6689 - val_accuracy: 0.9613 - val_loss: 0.7516
Epoch 37/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9930 - loss: 0.6660 - val_accuracy: 0.9598 - val_loss: 0.7585
Epoch 38/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 93ms/step - accuracy: 0.9945 - loss: 0.6607 - val_accuracy: 0.9784 - val_loss: 0.7131
Epoch 39/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 89ms/step - accuracy: 0.9966 - loss: 0.6544 - val_accuracy: 0.9740 - val_loss: 0.7160
Epoch 40/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9956 - loss: 0.6576 - val_accuracy: 0.9717 - val_loss: 0.7370
Epoch 41/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9947 - loss: 0.6596 - val_accuracy: 0.9747 - val_loss: 0.7156
Epoch 42/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9953 - loss: 0.6571 - val_accuracy: 0.9717 - val_loss: 0.7352
Epoch 43/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9939 - loss: 0.6584 - val_accuracy: 0.9725 - val_loss: 0.7174
Epoch 44/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9959 - loss: 0.6554 - val_accuracy: 0.9784 - val_loss: 0.7097
Epoch 45/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 92ms/step - accuracy: 0.9952 - loss: 0.6570 - val_accuracy: 0.9658 - val_loss: 0.7538
Epoch 46/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 92ms/step - accuracy: 0.9944 - loss: 0.6607 - val_accuracy: 0.9747 - val_loss: 0.7267
Epoch 47/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9959 - loss: 0.6572 - val_accuracy: 0.9658 - val_loss: 0.7484
Epoch 48/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9943 - loss: 0.6599 - val_accuracy: 0.9754 - val_loss: 0.7219
Epoch 49/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - accuracy: 0.9960 - loss: 0.6540 - val_accuracy: 0.9799 - val_loss: 0.6965
Epoch 50/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9970 - loss: 0.6500 - val_accuracy: 0.9688 - val_loss: 0.7253
Epoch 51/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9962 - loss: 0.6505 - val_accuracy: 0.9784 - val_loss: 0.7029
Epoch 52/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - accuracy: 0.9988 - loss: 0.6445 - val_accuracy: 0.9807 - val_loss: 0.6985
Epoch 53/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9982 - loss: 0.6468 - val_accuracy: 0.9754 - val_loss: 0.7118
Epoch 54/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9985 - loss: 0.6448 - val_accuracy: 0.9754 - val_loss: 0.7127
Epoch 55/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9995 - loss: 0.6417 - val_accuracy: 0.9807 - val_loss: 0.7125
Epoch 56/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9984 - loss: 0.6449 - val_accuracy: 0.9762 - val_loss: 0.7177
Epoch 57/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9972 - loss: 0.6479 - val_accuracy: 0.9807 - val_loss: 0.7035
Epoch 58/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9989 - loss: 0.6445 - val_accuracy: 0.9710 - val_loss: 0.7166
Epoch 59/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 93ms/step - accuracy: 0.9965 - loss: 0.6487 - val_accuracy: 0.9836 - val_loss: 0.6963
Epoch 60/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9966 - loss: 0.6485 - val_accuracy: 0.9777 - val_loss: 0.7066
Epoch 61/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9988 - loss: 0.6428 - val_accuracy: 0.9836 - val_loss: 0.6967
Epoch 62/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9992 - loss: 0.6419 - val_accuracy: 0.9807 - val_loss: 0.6960
Epoch 63/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9986 - loss: 0.6438 - val_accuracy: 0.9784 - val_loss: 0.7058
Epoch 64/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9985 - loss: 0.6423 - val_accuracy: 0.9814 - val_loss: 0.7052
Epoch 65/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 89ms/step - accuracy: 0.9986 - loss: 0.6430 - val_accuracy: 0.9799 - val_loss: 0.6949
Epoch 66/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 89ms/step - accuracy: 0.9981 - loss: 0.6424 - val_accuracy: 0.9792 - val_loss: 0.7015
Epoch 67/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9993 - loss: 0.6415 - val_accuracy: 0.9836 - val_loss: 0.6935
Epoch 68/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9995 - loss: 0.6398 - val_accuracy: 0.9814 - val_loss: 0.6903
Epoch 69/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9995 - loss: 0.6398 - val_accuracy: 0.9821 - val_loss: 0.6933
Epoch 70/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9993 - loss: 0.6402 - val_accuracy: 0.9836 - val_loss: 0.6864
Epoch 71/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9995 - loss: 0.6395 - val_accuracy: 0.9784 - val_loss: 0.7073
Epoch 72/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9983 - loss: 0.6429 - val_accuracy: 0.9829 - val_loss: 0.6946
Epoch 73/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - accuracy: 0.9986 - loss: 0.6414 - val_accuracy: 0.9844 - val_loss: 0.6945
Epoch 74/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9997 - loss: 0.6384 - val_accuracy: 0.9829 - val_loss: 0.6945
Epoch 75/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9996 - loss: 0.6392 - val_accuracy: 0.9821 - val_loss: 0.6965
Epoch 76/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9997 - loss: 0.6389 - val_accuracy: 0.9829 - val_loss: 0.6915
Epoch 77/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9989 - loss: 0.6404 - val_accuracy: 0.9829 - val_loss: 0.6899
Epoch 78/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9997 - loss: 0.6382 - val_accuracy: 0.9777 - val_loss: 0.7034
Epoch 79/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9995 - loss: 0.6388 - val_accuracy: 0.9844 - val_loss: 0.6902
Epoch 80/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9998 - loss: 0.6379 - val_accuracy: 0.9844 - val_loss: 0.6869
Epoch 81/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9997 - loss: 0.6382 - val_accuracy: 0.9844 - val_loss: 0.6887
Epoch 82/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9995 - loss: 0.6380 - val_accuracy: 0.9829 - val_loss: 0.6888
Epoch 83/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.9999 - loss: 0.6377 - val_accuracy: 0.9836 - val_loss: 0.6885
Epoch 84/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9998 - loss: 0.6375 - val_accuracy: 0.9829 - val_loss: 0.6926
Epoch 85/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9997 - loss: 0.6381 - val_accuracy: 0.9821 - val_loss: 0.6875
Epoch 86/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9998 - loss: 0.6375 - val_accuracy: 0.9821 - val_loss: 0.6903
Epoch 87/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9994 - loss: 0.6384 - val_accuracy: 0.9836 - val_loss: 0.6899
Epoch 88/100
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 91ms/step - accuracy: 0.9997 - loss: 0.6375 - val_accuracy: 0.9814 - val_loss: 0.6894
Epoch 88: early stopping
Restoring model weights from the end of the best epoch: 73.

  ✔ Done: best val acc = 98.44%  wall time = 1588s

╔══════════════════════════════════════════════════════════════════════╗
║  FINAL TEST-SET COMPARISON                                           ║
╠════════════════════════╦════════════╦════════════╦════════════╦══════╣
║  Model                 ║     Params ║   Test Acc ║   Macro F1 ║ Loss ║
╠════════════════════════╬════════════╬════════════╬════════════╬══════╣
║★ WhatNet-Arabic        ║ 5,423,110 ║     98.10%║     98.09%║0.701 ║
╚════════════════════════╩════════════╩════════════╩════════════╩══════╝

  ★  Winner: WhatNet-Arabic  (98.10% test accuracy)

[INFO] Results   → ./results/arabic_results.json
[INFO] Histories → ./results/arabic_histories.json

[DONE] AHCD Arabic benchmark complete.